# 17_generate_mel_nv_cue_qualitative_pdf

Generate qualitative GradCAM / Map Diff / FinerCAM comparison panels for the controlled binary MEL vs NV synthetic cue experiment.

This notebook uses the cued fixed center MEL/NV test set and compares:

1. Clean MEL/NV CE model
2. Cued MEL/NV CE model
3. Cued MEL/NV + HA model

The target/reference pair is class specific:

- GT = MEL: target MEL, reference NV
- GT = NV: target NV, reference MEL


In [ ]:
from pathlib import Path
import json
import subprocess
import shlex

import pandas as pd

QUAL_SEED = 42

REPO_ROOT = Path("..").resolve()
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"

SYN_ROOT = HAM_ROOT / "synthetic_cue" / f"mel_nv_fixed_center_seed{QUAL_SEED}"
CUE_CSV = SYN_ROOT / "csv" / f"ham_mel_nv_cue_on_mel_fixed_center_seed{QUAL_SEED}.csv"
CUE_QUAL_CSV = SYN_ROOT / "csv" / f"ham_mel_nv_cue_fixed_qualitative_10_seed{QUAL_SEED}.csv"

QUAL_ROOT = REPO_ROOT / "outputs" / f"qual_cue_mel_nv_seed{QUAL_SEED}"
QUAL_ROOT.mkdir(parents=True, exist_ok=True)

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

print("REPO_ROOT:", REPO_ROOT)
print("CUE_CSV:", CUE_CSV)
print("QUAL_ROOT:", QUAL_ROOT)


## 1. Create a small qualitative test CSV

This selects 5 MEL and 5 NV examples from the test split of the cued dataset.

For MEL images, the green cue is present. For NV images, no cue is present.

In [ ]:
df = pd.read_csv(CUE_CSV)

required_cols = [
    "image_id",
    "gt_label",
    "split",
    "image_rel_path",
    "mask_rel_path",
    "cue_applied",
    "cue_mask_rel_path",
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in {CUE_CSV}: {missing}")

test_df = df[df["split"] == "test"].copy()

mel = test_df[test_df["gt_label"] == "MEL"].sample(n=5, random_state=QUAL_SEED)
nv = test_df[test_df["gt_label"] == "NV"].sample(n=5, random_state=QUAL_SEED)

qual_df = pd.concat([mel, nv], axis=0).copy()
qual_df["order"] = qual_df["gt_label"].map({"MEL": 0, "NV": 1})
qual_df = qual_df.sort_values(["order", "image_id"]).drop(columns=["order"])

CUE_QUAL_CSV.parent.mkdir(parents=True, exist_ok=True)
qual_df.to_csv(CUE_QUAL_CSV, index=False)

print("Saved:", CUE_QUAL_CSV)
print("Rows:", len(qual_df))
print("Counts:")
print(qual_df.groupby(["gt_label", "cue_applied"]).size())

display(qual_df[[
    "image_id",
    "gt_label",
    "split",
    "image_rel_path",
    "mask_rel_path",
    "cue_applied",
    "cue_mask_rel_path",
]])


## 2. Define experiments

Check that these checkpoint paths exist. If your output folder names differ slightly, adjust the paths here.

In [ ]:
EXPERIMENTS_CUE = {
    "Clean CE": {
        "checkpoint": REPO_ROOT / "external" / "checkpoints2" / "checkpoint-best-clean.pth",
        "checkpoint_model_type": "panderm",
        "use_seg_gate": False,
        "out_dir": QUAL_ROOT / "cam_clean_ce_on_cued_test",
    },
    "Cue CE": {
        "checkpoint": REPO_ROOT / "external" / "checkpoints2" / "checkpoint-best-cue.pth",
        "checkpoint_model_type": "panderm",
        "use_seg_gate": False,
        "out_dir": QUAL_ROOT / "cam_cue_ce_on_cued_test",
    },
    "Cue HA": {
        "checkpoint": REPO_ROOT / "external" / "checkpoints2" / "checkpoint-best-cue-ha.pth",
        "checkpoint_model_type": "panderm",
        "use_seg_gate": False,
        "out_dir": QUAL_ROOT / "cam_cue_ha_on_cued_test",
    },
}

for name, cfg in EXPERIMENTS_CUE.items():
    print(name, "->", cfg["checkpoint"])
    if not cfg["checkpoint"].exists():
        print("  [WARN] missing checkpoint. Adjust path above.")


## 3. Helper functions

In [ ]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def generate_qualitative_cams(
    experiments: dict,
    csv_path: Path,
    img_dir: Path,
    num_samples: int,
    dry_run: bool = False,
):
    for exp_name, cfg in experiments.items():
        out_dir = cfg["out_dir"]
        out_dir.mkdir(parents=True, exist_ok=True)

        if cfg.get("use_seg_gate", False):
            panel_items = (
                "rgb_gt_mask,"
                "gate_weighted_gradcam_a,"
                "gate_weighted_gradcam_b,"
                "gate_weighted_map_diff,"
                "gate_weighted_finercam"
            )
        else:
            panel_items = (
                "rgb_gt_mask,"
                "gradcam_a,"
                "gradcam_b,"
                "map_diff,"
                "finercam"
            )

        cmd = [
            "python", "-m", "scripts.generate_finer_cam_panderm",
            "--csv", str(csv_path),
            "--image_col", "image_rel_path",
            "--img_dir", str(img_dir),
            "--gt_col", "gt_label",
            "--checkpoint", str(cfg["checkpoint"]),
            "--checkpoint_model_type", cfg["checkpoint_model_type"],
            "--class_names", "MEL,NV",
            "--out_dir", str(out_dir),
            "--num_samples", str(num_samples),
            "--method", "finercam",
            "--compare_mode", "gt_pair",
            "--A", "MEL",
            "--B", "NV",
            "--topk_compare", "1",
            "--alpha", "0.8",
            "--panel_items", panel_items,
            "--mask_root", str(MASK_ROOT),
            "--mask_col", "mask_rel_path",
            "--save_json",
            "--target_block_index", str(-4),
        ]

        if cfg.get("use_seg_gate", False):
            cmd += [
                "--use_seg_gate",
                "--seg_gate_bg_keep", str(cfg.get("seg_gate_bg_keep", 0.05)),
            ]

        print(f"\nRunning cue CAM generation: {exp_name}")
        run_command(cmd, dry_run=dry_run)


def build_qualitative_pdf(
    csv_path: Path,
    experiments_json_path: Path,
    out_pdf: Path,
    num_samples: int = 10,
    dry_run: bool = False,
):
    out_pdf.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "-m", "scripts.make_qualitative_comparison_pdf",
        "--csv", str(csv_path),
        "--image_col", "image_rel_path",
        "--gt_col", "gt_label",
        "--out_pdf", str(out_pdf),
        "--experiments_json_path", str(experiments_json_path),
        "--num_samples", str(num_samples),
        "--missing_policy", "placeholder",
    ]

    run_command(cmd, dry_run=dry_run)


## 4. Generate CAM panels

Set `dry_run=True` first if you only want to inspect the commands.

In [ ]:
generate_qualitative_cams(
    experiments=EXPERIMENTS_CUE,
    csv_path=CUE_QUAL_CSV,
    img_dir=IMG_DIR,
    num_samples=10,
    dry_run=False,
)


## 5. Write PDF config

In [ ]:
CONFIG_DIR = REPO_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

cue_pdf_config = [
    {"name": name, "folder": str(cfg["out_dir"].relative_to(REPO_ROOT))}
    for name, cfg in EXPERIMENTS_CUE.items()
]

CUE_JSON = CONFIG_DIR / f"qualitative_cue_mel_nv_3experiments_seed{QUAL_SEED}.json"
CUE_JSON.write_text(json.dumps(cue_pdf_config, indent=2))

print("Saved:", CUE_JSON)
print(json.dumps(cue_pdf_config, indent=2))


## 6. Build comparison PDF

In [ ]:
build_qualitative_pdf(
    csv_path=CUE_QUAL_CSV,
    experiments_json_path=CUE_JSON,
    out_pdf=QUAL_ROOT / f"qualitative_cue_mel_nv_3experiments_seed{QUAL_SEED}.pdf",
    num_samples=10,
    dry_run=False,
)


## Notes for interpretation

For MEL images, the green cue is present. The most important visual question is whether `Cue CE` and `Cue HA` place GradCAM / FinerCAM activation on that cue.

Expected pattern:

- `Clean CE`: no systematic focus on cue.
- `Cue CE`: likely strong focus on cue if the shortcut is learned.
- `Cue HA`: may also focus on cue because the cue is inside the lesion mask, so lesion HA does not penalize it.
